# DGGR Notebooks

This notebook is part of the **Deep Generative Genre Remastering (DGGR)** project.

## How To Run
- Prefer running from the repo root so relative paths resolve.
- Most notebooks assume you have the Lab artifacts under `saves/` and `saves2/` (ignored by git).
- See `docs/` for setup, data layout, and reproduction notes.

## Notes
- Outputs are intentionally stripped for version control cleanliness.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path


def _find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for _ in range(8):
        if (p / "dggr").exists():
            return p
        p = p.parent
    return (start or Path.cwd()).resolve()


REPO_ROOT = _find_repo_root()
DATA_ROOT = Path(os.environ.get("DGGR_DATA_ROOT", str(REPO_ROOT / "data")))
MANIFESTS_ROOT = Path(os.environ.get("DGGR_MANIFESTS_ROOT", str(DATA_ROOT / "_lab1_manifests")))

print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("MANIFESTS_ROOT:", MANIFESTS_ROOT)


# Lab 4: Structural Coherence & Long-Form Remastering

**Goal:** Assemble full-length style-transferred tracks from chunked generation while maintaining long-term structural consistency.

**Models:**
- **Diffusion V2** (epoch 6) — v-prediction DDIM with StyleAdaIN, BigVGAN vocoding (80 mel bands)
- **Neural Codec** (CodecLatentTranslator, run1055 stage 3) — EnCodec latent-space style transfer with neural decoding (24kHz)
- **Lab1 Encoder** — frozen encoder for z_content [128D] and z_style [128D] extraction

**Pipeline:**
1. Split source audio into overlapping ~3s chunks
2. Extract per-chunk features (mel, chroma, onset, beat, z_content, z_style)
3. Define target style trajectory (constant or interpolated)
4. Generate each chunk via Diffusion V2 DDIM
5. Assemble with cosine crossfade for seamless transitions
6. Compare with neural codec baseline for structural coherence analysis
7. Evaluate with objective metrics + listening tests

## 1. Setup & Imports

In [ ]:
from __future__ import annotations

import sys
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import IPython.display as ipd

# Add lab3 src to path
_SCRIPT_DIR = REPO_ROOT / "lab 3"
if str(_SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(_SCRIPT_DIR))

from src.lab3_diffusion_model import DiffusionUNetV2, EMA, NoiseSchedule
from src.lab3_diffusion_train import ddim_sample_v2, vocode_bigvgan
from src.lab3_diffusion_data import (
    DIFFUSION_SR, DIFFUSION_HOP, DIFFUSION_N_FFT, DIFFUSION_WIN,
    DIFFUSION_N_MELS, DIFFUSION_FMIN, DIFFUSION_FMAX,
    extract_bigvgan_mel_np, extract_chroma, extract_onset, extract_beat_grid,
    pad_or_trim, load_diffusion_cache, denormalize_mel,
)
from src.lab3_bridge import (
    FrozenLab1Encoder, extract_log_mel, fix_log_mel_frames,
    normalize_log_mel, denormalize_log_mel, load_audio_chunk,
)
from src.lab3_codec_models import CodecLatentTranslator
from src.lab3_codec_bridge import FrozenEncodec
from src.lab3_data import stratified_group_split_indices

print("Imports OK")

In [ ]:
# ---- Paths ----
BASE = REPO_ROOT
DIFFUSION_CKPT = BASE / "saves2" / "lab3_diffusion" / "run_d002" / "checkpoints" / "epoch_006.pt"
CODEC_CKPT = BASE / "saves2" / "lab3_codec_transfer" / "run1055" / "checkpoints" / "stage3_latest.pt"
LAB1_CKPT = BASE / "saves" / "lab1_run_combo_af_gate_exit_v2" / "latest.pt"
CACHE_DIR = BASE / "saves2" / "lab3_diffusion" / "run_d001" / "cache"
OUTPUT_DIR = BASE / "saves2" / "lab4_longform"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Diffusion ckpt exists: {DIFFUSION_CKPT.exists()}")
print(f"Codec ckpt exists: {CODEC_CKPT.exists()}")
print(f"Lab1 ckpt exists: {LAB1_CKPT.exists()}")
print(f"Cache dir exists: {CACHE_DIR.exists()}")

## 2. Load Models

In [ ]:
# ---- Lab1 Frozen Encoder ----
lab1_encoder = FrozenLab1Encoder(LAB1_CKPT, device=str(device))
print(f"Lab1 encoder loaded: {len(lab1_encoder.source_to_idx)} sources")
print(f"  Sources: {lab1_encoder.source_to_idx}")

In [ ]:
# ---- Diffusion V2 (epoch 6, best perceptual quality) ----
diff_model = DiffusionUNetV2(
    in_channels=15, out_channels=1,
    base_ch=64, ch_mults=(1, 2, 4, 4),
    n_res=2, attn_levels=(2, 3),
    z_content_dim=128, z_style_dim=128,
    dropout=0.1,
).to(device)

schedule = NoiseSchedule(T=1000).to(device)
ema = EMA(diff_model, decay=0.9999)

ckpt = torch.load(str(DIFFUSION_CKPT), map_location=device, weights_only=False)
diff_model.load_state_dict(ckpt["model"])
ema.load_state_dict(ckpt["ema"])
ema.shadow.eval()

n_params = sum(p.numel() for p in diff_model.parameters())
print(f"Diffusion V2 loaded: {n_params/1e6:.2f}M params")
print(f"  Epoch: {ckpt.get('epoch', '?')}, Step: {ckpt.get('global_step', '?')}")

In [ ]:
# ---- Neural Codec: CodecLatentTranslator + FrozenEncodec ----
# EnCodec for audio <-> latent encoding/decoding (24kHz)
encodec = FrozenEncodec(
    model_id="facebook/encodec_24khz",
    bandwidth=6.0,
    chunk_seconds=5.0,
    device=str(device),
)
CODEC_SR = encodec.cfg.sample_rate  # 24000
print(f"EnCodec loaded: sr={CODEC_SR}, bandwidth={encodec.cfg.bandwidth}")
print(f"  latent_channels={encodec.cfg.latent_channels}, frame_rate={encodec.cfg.frame_rate}")

# CodecLatentTranslator (style transfer in latent space)
codec_model = CodecLatentTranslator(
    in_channels=128,
    z_content_dim=128,
    z_style_dim=128,
    hidden_channels=256,
    n_blocks=10,
    noise_dim=32,
    residual_scale=0.5,
).to(device)

codec_ckpt = torch.load(str(CODEC_CKPT), map_location=device, weights_only=False)
codec_model.load_state_dict(codec_ckpt["generator"])
codec_model.eval()

n_params_codec = sum(p.numel() for p in codec_model.parameters())
print(f"CodecLatentTranslator loaded: {n_params_codec/1e6:.2f}M params, epoch={codec_ckpt.get('epoch', '?')}")

In [ ]:
# ---- BigVGAN Vocoder ----
import bigvgan as bvg
from huggingface_hub import hf_hub_download
import json as _json

# Workaround: bigvgan.from_pretrained() is incompatible with huggingface_hub>=1.0
# Load manually instead
_model_id = "nvidia/bigvgan_v2_22khz_80band_256x"
_config_path = hf_hub_download(_model_id, "config.json")
_ckpt_path = hf_hub_download(_model_id, "bigvgan_generator.pt")
with open(_config_path) as _f:
    _hparams = bvg.AttrDict(_json.load(_f))
vocoder = bvg.BigVGAN(_hparams)
vocoder.load_state_dict(torch.load(_ckpt_path, map_location="cpu", weights_only=False)["generator"])
vocoder.remove_weight_norm()
vocoder.eval().to(device)
print("BigVGAN vocoder loaded")

## 3. Load Cache & Genre Info

In [ ]:
index_df, arrays, genre_to_idx, cache_meta = load_diffusion_cache(CACHE_DIR, mmap=True)

MEL_MIN = cache_meta.mel_min
MEL_MAX = cache_meta.mel_max
idx_to_genre = {v: k for k, v in genre_to_idx.items()}

print(f"Cache: {cache_meta.n_samples} samples")
print(f"Mel range: [{MEL_MIN:.3f}, {MEL_MAX:.3f}]")
print(f"Genres: {genre_to_idx}")

# Split for val samples
genre_idx_arr = np.asarray(arrays["genre_idx"])
track_ids = index_df["track_id"].to_numpy()
train_idx, val_idx = stratified_group_split_indices(
    genre_idx_arr, track_ids, val_ratio=0.1, seed=328)
print(f"Val set: {len(val_idx)} samples")

## 4. Feature Extraction Utilities

In [ ]:
def normalize_diffusion_mel(mel: np.ndarray) -> np.ndarray:
    """Normalize BigVGAN log mel from [mel_min, mel_max] to [-1, 1]."""
    span = MEL_MAX - MEL_MIN
    return (2.0 * (mel - MEL_MIN) / span - 1.0).astype(np.float32)


def extract_chunk_features(
    audio: np.ndarray,
    sr: int = DIFFUSION_SR,
    n_frames: int = 256,
) -> dict:
    """Extract all features needed for one chunk of audio.
    
    Returns dict with:
        mel_80: [80, n_frames] BigVGAN log mel
        mel_80_norm: [1, 80, n_frames] normalized to [-1, 1]
        cond_feat: [14, 80, n_frames] chroma+onset+beat conditioning
        z_content: [128] content embedding
        z_style: [128] style embedding
        mel_96_db: [96, 256] Lab1 mel for GAN path
    """
    # BigVGAN mel (80 bands)
    mel_80 = extract_bigvgan_mel_np(audio, sr=sr)
    mel_80 = pad_or_trim(mel_80, n_frames, axis=1, pad_val=-11.5)
    mel_80_norm = normalize_diffusion_mel(mel_80)  # [80, n_frames]
    
    # Conditioning features
    chroma = extract_chroma(audio, sr=sr)
    chroma = pad_or_trim(chroma, n_frames, axis=1)
    onset = extract_onset(audio, sr=sr)
    onset = pad_or_trim(onset, n_frames, axis=0)
    beat = extract_beat_grid(audio, sr=sr, n_frames=n_frames)
    beat = pad_or_trim(beat, n_frames, axis=0)
    
    # Build cond_feat [14, 80, n_frames]
    H = 80
    chroma_exp = np.repeat(chroma[:, np.newaxis, :], H, axis=1)  # [12, 80, T]
    onset_exp = np.broadcast_to(onset[np.newaxis, np.newaxis, :], (1, H, n_frames))  # [1, 80, T]
    beat_exp = np.broadcast_to(beat[np.newaxis, np.newaxis, :], (1, H, n_frames))    # [1, 80, T]
    cond_feat = np.concatenate([chroma_exp, onset_exp, beat_exp], axis=0).astype(np.float32)
    
    # Lab1 encoder (96-band mel)
    log_mel_96 = extract_log_mel(audio, sr=sr)  # [96, T]
    log_mel_96 = fix_log_mel_frames(log_mel_96, n_frames=256)  # [96, 256]
    latent = lab1_encoder.infer_log_mel(log_mel_96)
    
    return {
        "mel_80": mel_80,
        "mel_80_norm": mel_80_norm[np.newaxis, :, :],  # [1, 80, T]
        "cond_feat": cond_feat,       # [14, 80, T]
        "z_content": latent["z_content"],  # [128]
        "z_style": latent["z_style"],      # [128]
        "mel_96_db": log_mel_96,            # [96, 256]
    }


print("Feature extraction utilities defined.")

## 5. Single-Chunk Generation (Diffusion V2 + Neural Codec)

In [ ]:
# Pick a validation sample from the cache
SAMPLE_IDX = val_idx[0]
sample_row = index_df.iloc[SAMPLE_IDX]
src_genre = sample_row["genre"]
print(f"Source sample: idx={SAMPLE_IDX}, genre={src_genre}")
print(f"  Path: {sample_row['path']}")

# Load source audio chunk
src_audio = load_audio_chunk(
    Path(sample_row["path"]),
    sample_rate=DIFFUSION_SR,
    seconds=3.0,  # ~256 frames
    start_sec=float(sample_row.get("start_sec", 0.0)),
)
print(f"  Audio shape: {src_audio.shape}, duration: {len(src_audio)/DIFFUSION_SR:.2f}s")

In [ ]:
# Extract features
feats = extract_chunk_features(src_audio, n_frames=256)
print("Features extracted:")
for k, v in feats.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: {v.shape}")

In [ ]:
# ---- Diffusion V2 single-chunk generation ----
# Use source content, pick a different target style
all_genres = sorted(genre_to_idx.keys())
target_genre = [g for g in all_genres if g != src_genre][0]
print(f"Style transfer: {src_genre} -> {target_genre}")

# Get a target style vector from a val sample of the target genre
tgt_genre_idx = genre_to_idx[target_genre]
tgt_candidates = val_idx[genre_idx_arr[val_idx] == tgt_genre_idx]
tgt_sample_idx = tgt_candidates[0]
z_style_target = np.array(arrays["z_style"][tgt_sample_idx]).astype(np.float32)
print(f"  Target style from sample idx={tgt_sample_idx}")

# Prepare tensors
cond_feat_t = torch.from_numpy(feats["cond_feat"]).unsqueeze(0).to(device)
z_c_t = torch.from_numpy(feats["z_content"]).unsqueeze(0).to(device)
z_s_t = torch.from_numpy(z_style_target).unsqueeze(0).to(device)

# DDIM sampling
with torch.no_grad():
    mel_gen = ddim_sample_v2(
        ema.shadow, schedule, cond_feat_t, z_c_t, z_s_t,
        n_steps=50, guidance_scale=2.0, device=device,
    )  # [1, 1, 80, 256]

print(f"Generated mel shape: {mel_gen.shape}")

# Vocode with BigVGAN
wav_gen = vocode_bigvgan(mel_gen, MEL_MIN, MEL_MAX, vocoder, device)  # [1, S]
print(f"Generated audio: {wav_gen.shape}, {wav_gen.shape[1]/DIFFUSION_SR:.2f}s")

In [ ]:
# ---- Neural Codec single-chunk generation ----
# Pipeline: audio -> resample to 24kHz -> EnCodec encode -> translate latents -> EnCodec decode -> resample back

# Resample source audio to codec sample rate (24kHz)
src_audio_24k = librosa.resample(src_audio, orig_sr=DIFFUSION_SR, target_sr=CODEC_SR, res_type="soxr_hq")
wav_tensor = torch.from_numpy(src_audio_24k).float().view(1, 1, -1).to(device)

# Encode to quantized embeddings
with torch.no_grad():
    q_emb = encodec.encode_embeddings(wav_tensor)  # [1, 128, T]
print(f"EnCodec embeddings: {q_emb.shape}")

# Run codec translator with source content + target style
z_c_codec = torch.from_numpy(feats["z_content"]).unsqueeze(0).to(device)
z_s_codec = torch.from_numpy(z_style_target).unsqueeze(0).to(device)

with torch.no_grad():
    noise = codec_model.sample_noise(1, device=device)
    q_translated = codec_model(q_emb, z_c_codec, z_s_codec, noise=noise)  # [1, 128, T]

# Decode back to audio
with torch.no_grad():
    wav_codec_24k = encodec.decode_embeddings(q_translated)  # [1, 1, S]
wav_codec_24k = wav_codec_24k[0, 0].cpu().numpy()

# Resample back to 22050Hz
wav_codec = librosa.resample(wav_codec_24k, orig_sr=CODEC_SR, target_sr=DIFFUSION_SR, res_type="soxr_hq")
# Normalize
peak = np.max(np.abs(wav_codec))
if peak > 0:
    wav_codec = wav_codec / peak * 0.95
print(f"Codec audio: {wav_codec.shape}, {len(wav_codec)/DIFFUSION_SR:.2f}s")

In [ ]:
# ---- Visualize & Listen ----
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Source mel (80-band)
axes[0].imshow(feats["mel_80"], origin="lower", aspect="auto", cmap="magma")
axes[0].set_title(f"Source ({src_genre}) - 80 band")

# Diffusion generated mel
mel_gen_np = denormalize_mel(
    mel_gen.squeeze().cpu(), MEL_MIN, MEL_MAX
).numpy()
axes[1].imshow(mel_gen_np, origin="lower", aspect="auto", cmap="magma")
axes[1].set_title(f"Diffusion V2 -> {target_genre}")

# Codec output mel (extract from output audio for visualization)
mel_codec_viz = librosa.feature.melspectrogram(y=wav_codec, sr=DIFFUSION_SR, n_mels=80, hop_length=256)
mel_codec_db = librosa.power_to_db(mel_codec_viz, ref=np.max)
axes[2].imshow(mel_codec_db, origin="lower", aspect="auto", cmap="magma")
axes[2].set_title(f"Neural Codec -> {target_genre}")

for ax in axes.flat:
    ax.set_xlabel("Time frames")
    ax.set_ylabel("Mel bands")

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "single_chunk_comparison.png"), dpi=150)
plt.show()

print("\n--- Source audio ---")
ipd.display(ipd.Audio(src_audio, rate=DIFFUSION_SR))
print(f"\n--- Diffusion V2 ({src_genre} -> {target_genre}) ---")
ipd.display(ipd.Audio(wav_gen[0], rate=DIFFUSION_SR))
print(f"\n--- Neural Codec ({src_genre} -> {target_genre}) ---")
ipd.display(ipd.Audio(wav_codec, rate=DIFFUSION_SR))

## 6. Long-Form Pipeline: Overlapping Windows + Cosine Crossfade

In [ ]:
def cosine_crossfade_weights(overlap_samples: int) -> np.ndarray:
    """Generate cosine crossfade weights for overlap region.
    
    Returns [overlap_samples] array from 1.0 -> 0.0 (fade-out for left chunk).
    Use (1 - weights) for the right chunk's fade-in.
    """
    t = np.linspace(0, np.pi / 2, overlap_samples)
    return np.cos(t).astype(np.float32) ** 2


def split_audio_overlapping(
    audio: np.ndarray,
    chunk_seconds: float = 3.0,
    overlap_seconds: float = 0.5,
    sr: int = DIFFUSION_SR,
) -> list[dict]:
    """Split audio into overlapping chunks.
    
    Returns list of dicts with 'audio', 'start_sample', 'end_sample'.
    """
    chunk_samples = int(chunk_seconds * sr)
    overlap_samples = int(overlap_seconds * sr)
    hop_samples = chunk_samples - overlap_samples
    
    chunks = []
    pos = 0
    while pos < len(audio):
        end = min(pos + chunk_samples, len(audio))
        chunk = audio[pos:end]
        # Pad if shorter than chunk_samples
        if len(chunk) < chunk_samples:
            chunk = np.pad(chunk, (0, chunk_samples - len(chunk)))
        chunks.append({
            "audio": chunk,
            "start_sample": pos,
            "end_sample": end,
        })
        if end >= len(audio):
            break
        pos += hop_samples
    
    return chunks


def assemble_audio_crossfade(
    chunk_wavs: list[np.ndarray],
    overlap_seconds: float = 0.5,
    sr: int = DIFFUSION_SR,
) -> np.ndarray:
    """Assemble generated chunks with cosine crossfade.
    
    Args:
        chunk_wavs: list of audio arrays, one per chunk
        overlap_seconds: overlap duration for crossfade
        sr: sample rate
    
    Returns:
        Assembled audio array.
    """
    if len(chunk_wavs) == 0:
        return np.array([], dtype=np.float32)
    if len(chunk_wavs) == 1:
        return chunk_wavs[0]
    
    overlap_samples = int(overlap_seconds * sr)
    
    # Start with first chunk
    result = list(chunk_wavs[0])
    
    for i in range(1, len(chunk_wavs)):
        chunk = chunk_wavs[i]
        fade = cosine_crossfade_weights(overlap_samples)
        
        # Crossfade the overlap region
        overlap_start = len(result) - overlap_samples
        for j in range(overlap_samples):
            if overlap_start + j < len(result):
                result[overlap_start + j] = (
                    result[overlap_start + j] * fade[j] +
                    chunk[j] * (1.0 - fade[j])
                )
        
        # Append the non-overlapping part
        result.extend(chunk[overlap_samples:])
    
    return np.array(result, dtype=np.float32)


# Test
test_fade = cosine_crossfade_weights(100)
plt.plot(test_fade, label="fade-out (left)")
plt.plot(1.0 - test_fade, label="fade-in (right)")
plt.title("Cosine crossfade weights")
plt.legend()
plt.show()
print("Crossfade utilities defined.")

## 7. Style Trajectory Interpolation

In [ ]:
def build_style_trajectory(
    n_chunks: int,
    z_style_start: np.ndarray,
    z_style_end: np.ndarray | None = None,
    mode: str = "constant",
) -> list[np.ndarray]:
    """Build a style trajectory across chunks.
    
    Modes:
        'constant': Use z_style_start for all chunks
        'linear': Linear interpolation from start to end
        'cosine': Cosine interpolation (smoother transitions)
    
    Returns list of z_style vectors, one per chunk.
    """
    if mode == "constant" or z_style_end is None:
        return [z_style_start.copy() for _ in range(n_chunks)]
    
    trajectory = []
    for i in range(n_chunks):
        t = float(i) / max(1, n_chunks - 1)
        if mode == "cosine":
            # Smoother interpolation
            t = 0.5 * (1.0 - np.cos(np.pi * t))
        z = (1.0 - t) * z_style_start + t * z_style_end
        # Re-normalize (L2) to stay on the style manifold
        z = z / (np.linalg.norm(z) + 1e-8)
        trajectory.append(z.astype(np.float32))
    
    return trajectory


def build_multi_keyframe_trajectory(
    n_chunks: int,
    keyframes: list[tuple[int, np.ndarray]],
    interp_mode: str = "cosine",
) -> list[np.ndarray]:
    """Build style trajectory from multiple keyframes.
    
    Args:
        n_chunks: total number of chunks
        keyframes: list of (chunk_index, z_style) pairs
        interp_mode: 'linear' or 'cosine'
    
    Returns list of z_style vectors.
    """
    keyframes = sorted(keyframes, key=lambda x: x[0])
    trajectory = []
    
    for i in range(n_chunks):
        # Find surrounding keyframes
        left_kf = keyframes[0]
        right_kf = keyframes[-1]
        for j in range(len(keyframes) - 1):
            if keyframes[j][0] <= i <= keyframes[j + 1][0]:
                left_kf = keyframes[j]
                right_kf = keyframes[j + 1]
                break
        
        if left_kf[0] == right_kf[0]:
            z = left_kf[1].copy()
        else:
            t = float(i - left_kf[0]) / float(right_kf[0] - left_kf[0])
            t = np.clip(t, 0.0, 1.0)
            if interp_mode == "cosine":
                t = 0.5 * (1.0 - np.cos(np.pi * t))
            z = (1.0 - t) * left_kf[1] + t * right_kf[1]
        
        z = z / (np.linalg.norm(z) + 1e-8)
        trajectory.append(z.astype(np.float32))
    
    return trajectory


print("Style trajectory utilities defined.")

## 8. Full Long-Form Generation Pipeline

In [ ]:
@torch.no_grad()
def generate_longform_diffusion(
    source_audio: np.ndarray,
    style_trajectory: list[np.ndarray],
    chunk_seconds: float = 3.0,
    overlap_seconds: float = 0.5,
    ddim_steps: int = 50,
    guidance_scale: float = 2.0,
    n_frames: int = 256,
    sr: int = DIFFUSION_SR,
    verbose: bool = True,
) -> tuple[np.ndarray, list[dict]]:
    """Generate a full-length style-transferred track using Diffusion V2.
    
    Args:
        source_audio: full source audio array
        style_trajectory: list of z_style [128] vectors, one per chunk
        chunk_seconds: chunk duration
        overlap_seconds: overlap for crossfade
        ddim_steps: DDIM sampling steps
        guidance_scale: CFG scale
        n_frames: mel frames per chunk
        sr: sample rate
        verbose: print progress
    
    Returns:
        (assembled_audio, chunk_info_list)
    """
    chunks = split_audio_overlapping(source_audio, chunk_seconds, overlap_seconds, sr)
    n_chunks = len(chunks)
    
    # Extend trajectory if needed
    while len(style_trajectory) < n_chunks:
        style_trajectory.append(style_trajectory[-1].copy())
    
    chunk_wavs = []
    chunk_infos = []
    
    for i, chunk_data in enumerate(chunks):
        if verbose:
            print(f"  Generating chunk {i+1}/{n_chunks}...", end="", flush=True)
        
        # Extract features
        feats = extract_chunk_features(chunk_data["audio"], sr=sr, n_frames=n_frames)
        
        # Prepare tensors
        cond_feat_t = torch.from_numpy(feats["cond_feat"]).unsqueeze(0).to(device)
        z_c = torch.from_numpy(feats["z_content"]).unsqueeze(0).to(device)
        z_s = torch.from_numpy(style_trajectory[i]).unsqueeze(0).to(device)
        
        # DDIM generate
        mel_gen = ddim_sample_v2(
            ema.shadow, schedule, cond_feat_t, z_c, z_s,
            n_steps=ddim_steps, guidance_scale=guidance_scale, device=device,
        )
        
        # Vocode
        wav = vocode_bigvgan(mel_gen, MEL_MIN, MEL_MAX, vocoder, device)
        chunk_wavs.append(wav[0])
        
        chunk_infos.append({
            "chunk_idx": i,
            "z_content": feats["z_content"],
            "z_style_target": style_trajectory[i],
            "mel_gen_shape": mel_gen.shape,
        })
        
        if verbose:
            print(f" done ({wav[0].shape[0]/sr:.2f}s)")
    
    # Assemble with crossfade
    assembled = assemble_audio_crossfade(chunk_wavs, overlap_seconds, sr)
    
    # Normalize
    peak = np.max(np.abs(assembled))
    if peak > 0:
        assembled = assembled / peak * 0.95
    
    if verbose:
        print(f"\nAssembled: {len(assembled)/sr:.2f}s from {n_chunks} chunks")
    
    return assembled, chunk_infos


@torch.no_grad()
def generate_longform_codec(
    source_audio: np.ndarray,
    style_trajectory: list[np.ndarray],
    chunk_seconds: float = 3.0,
    overlap_seconds: float = 0.5,
    sr: int = DIFFUSION_SR,
    verbose: bool = True,
) -> np.ndarray:
    """Generate a full-length style-transferred track using neural codec.
    
    Pipeline per chunk:
      audio (22050Hz) -> resample 24kHz -> EnCodec encode -> CodecLatentTranslator
      -> EnCodec decode -> resample 22050Hz
    """
    chunks = split_audio_overlapping(source_audio, chunk_seconds, overlap_seconds, sr)
    n_chunks = len(chunks)
    
    while len(style_trajectory) < n_chunks:
        style_trajectory.append(style_trajectory[-1].copy())
    
    chunk_wavs = []
    for i, chunk_data in enumerate(chunks):
        if verbose:
            print(f"  Codec chunk {i+1}/{n_chunks}...", end="", flush=True)
        
        # Extract Lab1 features (at 22050Hz)
        feats = extract_chunk_features(chunk_data["audio"], sr=sr, n_frames=256)
        
        # Resample to codec sample rate (24kHz) and encode
        audio_24k = librosa.resample(
            chunk_data["audio"], orig_sr=sr, target_sr=CODEC_SR, res_type="soxr_hq")
        wav_t = torch.from_numpy(audio_24k).float().view(1, 1, -1).to(device)
        q_emb = encodec.encode_embeddings(wav_t)  # [1, 128, T]
        
        # Translate latents
        z_c = torch.from_numpy(feats["z_content"]).unsqueeze(0).to(device)
        z_s = torch.from_numpy(style_trajectory[i]).unsqueeze(0).to(device)
        noise = codec_model.sample_noise(1, device=device)
        q_translated = codec_model(q_emb, z_c, z_s, noise=noise)
        
        # Decode and resample back
        wav_out_24k = encodec.decode_embeddings(q_translated)[0, 0].cpu().numpy()
        wav_out = librosa.resample(
            wav_out_24k, orig_sr=CODEC_SR, target_sr=sr, res_type="soxr_hq")
        
        # Trim to match expected chunk length
        expected_len = int(chunk_seconds * sr)
        if len(wav_out) > expected_len:
            wav_out = wav_out[:expected_len]
        elif len(wav_out) < expected_len:
            wav_out = np.pad(wav_out, (0, expected_len - len(wav_out)))
        
        chunk_wavs.append(wav_out.astype(np.float32))
        if verbose:
            print(f" done")
    
    assembled = assemble_audio_crossfade(chunk_wavs, overlap_seconds, sr)
    peak = np.max(np.abs(assembled))
    if peak > 0:
        assembled = assembled / peak * 0.95
    
    if verbose:
        print(f"Codec assembled: {len(assembled)/sr:.2f}s from {n_chunks} chunks")
    
    return assembled


print("Long-form generation pipelines defined.")

## 9. Demo: Long-Form Generation with Style Transfer

In [ ]:
# Load a longer source audio (~15-30 seconds)
# Pick a val sample and load more of it
demo_idx = val_idx[5]  # pick a sample
demo_row = index_df.iloc[demo_idx]
demo_path = Path(demo_row["path"])
demo_genre = demo_row["genre"]
print(f"Demo source: {demo_path.name}")
print(f"  Genre: {demo_genre}")

# Load up to 15 seconds
demo_audio = load_audio_chunk(
    demo_path, sample_rate=DIFFUSION_SR,
    seconds=15.0, start_sec=0.0,
)
print(f"  Duration: {len(demo_audio)/DIFFUSION_SR:.2f}s")

print("\n--- Source audio ---")
ipd.display(ipd.Audio(demo_audio, rate=DIFFUSION_SR))

In [ ]:
# ---- Constant style transfer (all chunks same target style) ----
target_genre_name = [g for g in all_genres if g != demo_genre][0]
target_g_idx = genre_to_idx[target_genre_name]
tgt_cands = val_idx[genre_idx_arr[val_idx] == target_g_idx]
z_style_tgt = np.array(arrays["z_style"][tgt_cands[0]]).astype(np.float32)

print(f"=== Constant style transfer: {demo_genre} -> {target_genre_name} ===")
print(f"Using target style from val sample {tgt_cands[0]}")

# Build constant trajectory
n_chunks_est = int(np.ceil(len(demo_audio) / (DIFFUSION_SR * 2.5)))  # approximate
trajectory_const = build_style_trajectory(n_chunks_est + 5, z_style_tgt, mode="constant")

wav_diff_const, infos_const = generate_longform_diffusion(
    demo_audio, trajectory_const,
    chunk_seconds=3.0, overlap_seconds=0.5,
    ddim_steps=50, guidance_scale=2.0,
)

sf.write(str(OUTPUT_DIR / "demo_diffusion_constant.wav"), wav_diff_const, DIFFUSION_SR)
print(f"\nSaved: {OUTPUT_DIR / 'demo_diffusion_constant.wav'}")
print(f"\n--- Diffusion V2 (constant {target_genre_name}) ---")
ipd.display(ipd.Audio(wav_diff_const, rate=DIFFUSION_SR))

In [ ]:
# ---- Interpolated style transfer (gradual genre morph) ----
genre_a = all_genres[0]
genre_b = all_genres[-1]
cands_a = val_idx[genre_idx_arr[val_idx] == genre_to_idx[genre_a]]
cands_b = val_idx[genre_idx_arr[val_idx] == genre_to_idx[genre_b]]
z_style_a = np.array(arrays["z_style"][cands_a[0]]).astype(np.float32)
z_style_b = np.array(arrays["z_style"][cands_b[0]]).astype(np.float32)

print(f"=== Interpolated style: {genre_a} -> {genre_b} (cosine) ===")

trajectory_interp = build_style_trajectory(
    n_chunks_est + 5, z_style_a, z_style_b, mode="cosine")

wav_diff_interp, infos_interp = generate_longform_diffusion(
    demo_audio, trajectory_interp,
    chunk_seconds=3.0, overlap_seconds=0.5,
    ddim_steps=50, guidance_scale=2.0,
)

sf.write(str(OUTPUT_DIR / "demo_diffusion_interpolated.wav"), wav_diff_interp, DIFFUSION_SR)
print(f"\nSaved: {OUTPUT_DIR / 'demo_diffusion_interpolated.wav'}")
print(f"\n--- Diffusion V2 (interpolated {genre_a}->{genre_b}) ---")
ipd.display(ipd.Audio(wav_diff_interp, rate=DIFFUSION_SR))

In [ ]:
# ---- Neural Codec baseline for comparison ----
print(f"=== Neural Codec: {demo_genre} -> {target_genre_name} ===")

# Use same target style trajectory as diffusion constant
trajectory_codec = build_style_trajectory(n_chunks_est + 5, z_style_tgt, mode="constant")

wav_codec_long = generate_longform_codec(
    demo_audio, trajectory_codec,
    chunk_seconds=3.0, overlap_seconds=0.5,
)

sf.write(str(OUTPUT_DIR / "demo_codec_constant.wav"), wav_codec_long, DIFFUSION_SR)
print(f"\n--- Neural Codec (constant {target_genre_name}) ---")
ipd.display(ipd.Audio(wav_codec_long, rate=DIFFUSION_SR))

## 10. Evaluation Metrics

In [ ]:
from src.lab3_diffusion_train import pitch_correlation

def mel_spectral_distance(audio_a: np.ndarray, audio_b: np.ndarray, sr: int = DIFFUSION_SR) -> float:
    """L1 distance between log mel spectrograms (content preservation proxy)."""
    mel_a = librosa.feature.melspectrogram(y=audio_a, sr=sr, n_mels=80, hop_length=256)
    mel_b = librosa.feature.melspectrogram(y=audio_b, sr=sr, n_mels=80, hop_length=256)
    mel_a_db = librosa.power_to_db(mel_a, ref=np.max)
    mel_b_db = librosa.power_to_db(mel_b, ref=np.max)
    n = min(mel_a_db.shape[1], mel_b_db.shape[1])
    return float(np.mean(np.abs(mel_a_db[:, :n] - mel_b_db[:, :n])))


def chunk_boundary_discontinuity(
    audio: np.ndarray,
    chunk_seconds: float = 3.0,
    overlap_seconds: float = 0.5,
    sr: int = DIFFUSION_SR,
    window_ms: float = 50.0,
) -> list[float]:
    """Measure spectral discontinuity at chunk boundaries.
    
    For each boundary, computes the L1 distance between mel frames
    just before and just after the boundary center.
    Lower = smoother transition.
    """
    hop_samples = int((chunk_seconds - overlap_seconds) * sr)
    window_frames = max(1, int(window_ms / 1000 * sr / 256))
    
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=80, hop_length=256)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    discontinuities = []
    boundary_sample = hop_samples
    while boundary_sample < len(audio) - hop_samples:
        frame = boundary_sample // 256
        if frame - window_frames >= 0 and frame + window_frames < mel_db.shape[1]:
            left = mel_db[:, frame - window_frames:frame].mean(axis=1)
            right = mel_db[:, frame:frame + window_frames].mean(axis=1)
            disc = float(np.mean(np.abs(left - right)))
            discontinuities.append(disc)
        boundary_sample += hop_samples
    
    return discontinuities


print("Evaluation metrics defined.")

In [ ]:
# ---- Evaluate all outputs ----
min_len = min(len(demo_audio), len(wav_diff_const), len(wav_diff_interp), len(wav_codec_long))

results = {}
for name, wav in [
    ("Diffusion (constant)", wav_diff_const),
    ("Diffusion (interpolated)", wav_diff_interp),
    ("Codec (constant)", wav_codec_long),
]:
    # Pitch correlation with source (content preservation)
    pc = pitch_correlation(demo_audio[:min_len], wav[:min_len], sr=DIFFUSION_SR)
    
    # Mel spectral distance (lower = more similar to source)
    msd = mel_spectral_distance(demo_audio[:min_len], wav[:min_len])
    
    # Chunk boundary discontinuity (lower = smoother)
    disc = chunk_boundary_discontinuity(wav)
    avg_disc = float(np.mean(disc)) if disc else 0.0
    
    results[name] = {
        "pitch_corr": pc,
        "mel_spectral_dist": msd,
        "avg_boundary_disc": avg_disc,
        "n_boundaries": len(disc),
    }
    print(f"\n{name}:")
    print(f"  Pitch correlation:   {pc:.4f}")
    print(f"  Mel spectral dist:   {msd:.2f} dB")
    print(f"  Avg boundary disc:   {avg_disc:.2f} dB ({len(disc)} boundaries)")

In [ ]:
# ---- Summary bar chart ----
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

names = list(results.keys())
x = np.arange(len(names))

# Pitch correlation (higher = better content preservation)
vals = [results[n]["pitch_corr"] for n in names]
axes[0].bar(x, vals, color=["steelblue", "coral", "green"])
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=15, ha="right", fontsize=8)
axes[0].set_ylabel("Correlation")
axes[0].set_title("Pitch Correlation (higher=better)")
axes[0].set_ylim(0, 1)

# Mel spectral distance (lower for same style, but style transfer changes it)
vals = [results[n]["mel_spectral_dist"] for n in names]
axes[1].bar(x, vals, color=["steelblue", "coral", "green"])
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=15, ha="right", fontsize=8)
axes[1].set_ylabel("L1 (dB)")
axes[1].set_title("Mel Spectral Distance")

# Boundary discontinuity (lower = smoother)
vals = [results[n]["avg_boundary_disc"] for n in names]
axes[2].bar(x, vals, color=["steelblue", "coral", "green"])
axes[2].set_xticks(x)
axes[2].set_xticklabels(names, rotation=15, ha="right", fontsize=8)
axes[2].set_ylabel("L1 (dB)")
axes[2].set_title("Boundary Discontinuity (lower=smoother)")

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "evaluation_summary.png"), dpi=150)
plt.show()

## 11. Waveform & Spectrogram Visualization

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 12))

for ax, (name, wav) in zip(axes, [
    ("Source", demo_audio),
    (f"Diffusion V2 (constant -> {target_genre_name})", wav_diff_const),
    (f"Diffusion V2 (interpolated {genre_a}->{genre_b})", wav_diff_interp),
    (f"Neural Codec (constant -> {target_genre_name})", wav_codec_long),
]):
    mel_spec = librosa.feature.melspectrogram(y=wav[:min_len], sr=DIFFUSION_SR, n_mels=80, hop_length=256)
    mel_db = librosa.power_to_db(mel_spec, ref=np.max)
    img = ax.imshow(mel_db, origin="lower", aspect="auto", cmap="magma",
                    extent=[0, min_len / DIFFUSION_SR, 0, 80])
    ax.set_title(name)
    ax.set_ylabel("Mel band")
    plt.colorbar(img, ax=ax, fraction=0.02, pad=0.01)

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "spectrogram_comparison.png"), dpi=150)
plt.show()

## 12. Style Classification Audit

Use the frozen Lab1 encoder to classify the style of generated chunks and verify that the style transfer actually took effect.

In [ ]:
def audit_style_per_chunk(
    audio: np.ndarray,
    chunk_seconds: float = 3.0,
    overlap_seconds: float = 0.5,
    sr: int = DIFFUSION_SR,
) -> list[dict]:
    """Classify each chunk's style using Lab1 encoder."""
    chunks = split_audio_overlapping(audio, chunk_seconds, overlap_seconds, sr)
    source_to_idx = lab1_encoder.source_to_idx
    idx_to_source = {v: k for k, v in source_to_idx.items()}
    
    results = []
    for i, ch in enumerate(chunks):
        log_mel = extract_log_mel(ch["audio"], sr=sr)
        log_mel = fix_log_mel_frames(log_mel, n_frames=256)
        
        x = torch.from_numpy(log_mel).unsqueeze(0).to(lab1_encoder.device)
        out = lab1_encoder.model(x, grl_lambda=0.0)
        probs = torch.softmax(out["style_logits"][0], dim=0).cpu().numpy()
        pred_idx = int(np.argmax(probs))
        pred_source = idx_to_source.get(pred_idx, str(pred_idx))
        
        results.append({
            "chunk": i,
            "predicted_style": pred_source,
            "confidence": float(probs[pred_idx]),
            "probs": {idx_to_source.get(j, str(j)): float(probs[j]) for j in range(len(probs))},
        })
    
    return results


# Audit diffusion constant output
print(f"Style audit: Diffusion constant -> {target_genre_name}")
audit_const = audit_style_per_chunk(wav_diff_const)
for r in audit_const:
    print(f"  Chunk {r['chunk']}: pred={r['predicted_style']} (conf={r['confidence']:.3f})")

print(f"\nStyle audit: Diffusion interpolated ({genre_a} -> {genre_b})")
audit_interp = audit_style_per_chunk(wav_diff_interp)
for r in audit_interp:
    print(f"  Chunk {r['chunk']}: pred={r['predicted_style']} (conf={r['confidence']:.3f})")

In [ ]:
# Visualize style trajectory
if audit_interp:
    sources = sorted(lab1_encoder.source_to_idx.keys())
    chunks_x = [r["chunk"] for r in audit_interp]
    
    fig, ax = plt.subplots(figsize=(12, 4))
    for src in sources:
        probs = [r["probs"].get(src, 0.0) for r in audit_interp]
        ax.plot(chunks_x, probs, marker="o", label=src)
    
    ax.set_xlabel("Chunk index")
    ax.set_ylabel("Style probability")
    ax.set_title(f"Style trajectory: {genre_a} -> {genre_b}")
    ax.legend()
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / "style_trajectory.png"), dpi=150)
    plt.show()

## 13. Batch Generation: All Genre Pairs

In [ ]:
# Generate one example per genre pair
batch_dir = OUTPUT_DIR / "batch_samples"
batch_dir.mkdir(parents=True, exist_ok=True)

batch_results = []

for src_g in all_genres:
    src_g_idx = genre_to_idx[src_g]
    src_cands = val_idx[genre_idx_arr[val_idx] == src_g_idx]
    if len(src_cands) == 0:
        continue
    
    # Load 10 seconds of source
    src_row = index_df.iloc[src_cands[0]]
    src_wav = load_audio_chunk(
        Path(src_row["path"]), sample_rate=DIFFUSION_SR,
        seconds=10.0, start_sec=0.0,
    )
    
    for tgt_g in all_genres:
        if tgt_g == src_g:
            continue
        
        tgt_g_idx = genre_to_idx[tgt_g]
        tgt_cands = val_idx[genre_idx_arr[val_idx] == tgt_g_idx]
        if len(tgt_cands) == 0:
            continue
        
        z_s = np.array(arrays["z_style"][tgt_cands[0]]).astype(np.float32)
        traj = build_style_trajectory(20, z_s, mode="constant")
        
        print(f"\n--- {src_g} -> {tgt_g} ---")
        wav_out, _ = generate_longform_diffusion(
            src_wav, traj,
            chunk_seconds=3.0, overlap_seconds=0.5,
            ddim_steps=50, guidance_scale=2.0,
        )
        
        out_path = batch_dir / f"{src_g}_to_{tgt_g}.wav"
        sf.write(str(out_path), wav_out, DIFFUSION_SR)
        
        pc = pitch_correlation(src_wav[:len(wav_out)], wav_out, sr=DIFFUSION_SR)
        disc = chunk_boundary_discontinuity(wav_out)
        
        batch_results.append({
            "source": src_g, "target": tgt_g,
            "pitch_corr": pc,
            "avg_disc": float(np.mean(disc)) if disc else 0.0,
            "path": str(out_path),
        })
        print(f"  pitch_corr={pc:.4f}, avg_disc={np.mean(disc) if disc else 0:.2f}")

# Save summary
import pandas as pd
batch_df = pd.DataFrame(batch_results)
batch_df.to_csv(batch_dir / "batch_summary.csv", index=False)
print(f"\nBatch complete: {len(batch_results)} pairs")
print(batch_df.to_string(index=False))

## 14. Summary & Conclusions

### Key Findings

1. **Diffusion V2 (epoch 6)** produces the highest perceptual quality per chunk, with v-prediction and StyleAdaIN providing clean style transfer.

2. **Overlapping windows with cosine crossfade** eliminate audible seams at chunk boundaries — the boundary discontinuity metric shows smooth transitions.

3. **Style trajectory interpolation** enables smooth gradual style morphing across the track, as confirmed by the per-chunk style classification audit.

4. **Neural Codec (CodecLatentTranslator)** operates directly in EnCodec's latent space, preserving source structure while modifying timbre. Uses neural decoding (no Griffin-Lim artifacts).

### Architecture Summary

| Component | Details |
|---|---|
| Diffusion V2 | 64-base UNet, [1,2,4,4] channels, v-prediction, StyleAdaIN |
| Noise Schedule | Cosine, T=1000, DDIM 50 steps |
| Vocoder | BigVGAN v2 22kHz 80-band |
| Neural Codec | CodecLatentTranslator (10 FiLM blocks, 256 hidden) + EnCodec 24kHz |
| Encoder | Lab1 ChunkEncoder, 128D z_content + 128D z_style |
| Assembly | 3s chunks, 0.5s overlap, cosine crossfade |